# Importing Databricks Data into Neptune Analytics via Athena Federation
This notebook demonstrates how to connect PaySim data stored in Databricks and, using Athena Federated Query, create a graph view of the data in Neptune Analytics. 


### Prerequisite

To enable querying Databricks connector from Amazon Athena, the Athena Databricks Connector must first be deployed in your AWS account.
Deployment and setup instructions are available in the following resources:

**Installation guide**

../connectors/athena-databricks-connector/README.md


### What this notebook covers:
1. Download the [Kaggle PaySim1 dataset](https://www.kaggle.com/datasets/ealaxi/paysim1), a synthetic financial dataset simulating mobile money transactions, and upload it to Databricks.

2. Use Amazon Athena to federate queries against the Databricks table and generate vertex/edge projections
3. Import the projections into Amazon Neptune Analytics
4. Run community detection (Louvain) to identify transaction clusters


## Setup

Import the necessary libraries and set up logging.

In [ ]:
%pip install databricks-sdk databricks-sql-connector kagglehub

In [ ]:
import dotenv
import kagglehub
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks import sql

from nx_neptune import empty_s3_bucket, instance_management, NeptuneGraph, set_config_graph_id
from nx_neptune.instance_management import execute_athena_query, _clean_s3_path, get_athena_query_results
from nx_neptune.utils.utils import get_stdout_logger, validate_and_get_env


# Configure logging to see detailed information about the instance creation process
logger = get_stdout_logger(__name__, [
    'nx_neptune.instance_management',
    'nx_neptune.utils.task_future',
    'nx_neptune.interface',
    __name__
])

## Configuration

Check for environment variables necessary for the notebook.

In [ ]:
# Required environment variables for Neptune Analytics and Athena                           
dotenv.load_dotenv()                                                                        
env_vars = validate_and_get_env([                                                           
  'NETWORKX_S3_DATA_LAKE_BUCKET_PATH',                                                    
  'NETWORKX_S3_NA_IMPORT_BUCKET_PATH',                                                    
  'NETWORKX_S3_LOG_BUCKET_PATH',                                                          
  'NETWORKX_S3_TABLES_DATABASE',                                                          
  'NETWORKX_S3_TABLES_TABLENAME',                                                         
  'NETWORKX_GRAPH_ID',                                                                    
])                                                                                          
                                                                                          
(s3_location_data_lake, s3_location_na_import, s3_location_log,                             
s3_tables_database, s3_tables_tablename, graph_id) = env_vars.values()                     
                                                                                          
# Optional — only needed to upload test data to Databricks (skip if table already exists)   
db_env = validate_and_get_env([                                                             
  'DATABRICKS_HOST',                                                                      
  'DATABRICKS_TOKEN',                                                                     
  'DATABRICKS_HTTP_PATH',                                                                 
])                                                                                          
                                                                                          
db_host, db_token, db_http_path = db_env.values()                                           

## Test Data Setup

The transaction dataset used in this demo is sourced from [Kaggle PaySim1 dataset](https://www.kaggle.com/datasets/ealaxi/paysim1)
, a synthetic financial dataset simulating mobile money
transactions.

The setup cell below automates the full data ingestion pipeline:

1. Download — The dataset is fetched programmatically using the kagglehub package, which 
handles caching to avoid redundant downloads on subsequent runs.

2. Upload to Databricks Volume — The CSV file is uploaded to a Unity Catalog Volume via the 
Databricks SDK, staging it in cloud storage accessible by the SQL Warehouse.

3. Create Delta Table — A `CREATE TABLE AS SELECT` statement reads the CSV from the Volume and 
materializes it as a managed Delta table. Schema is inferred automatically from the CSV 
headers.

If the table already exists, all three steps are skipped entirely.

In [ ]:
volume_path = "/Volumes/workspace/default/demo"
table_name = "workspace.default.paysim_transactions"

with sql.connect(server_hostname=db_host, http_path=db_http_path, access_token=db_token) as conn:
    cursor = conn.cursor()
    cursor.execute(f"SHOW TABLES IN workspace.default LIKE 'paysim_transactions'")
    if cursor.fetchone():
        print(f"{table_name} already exists, skipping")
    else:
        print("Downloading PaySim dataset from Kaggle...") 
        paysim_path = Path(kagglehub.dataset_download("ealaxi/paysim1"))
        csv_file = next(paysim_path.glob("*.csv"))

        print(f"Uploading {csv_file.name} to Databricks Volume...")
        w = WorkspaceClient(host=f"https://{db_host}", token=db_token)
        with open(csv_file, "rb") as f:
            w.files.upload(f"{volume_path}/{csv_file.name}", f, overwrite=True)

        print(f"Creating Delta table {table_name}...")  
        cursor.execute(f"""
            CREATE TABLE {table_name} AS
            SELECT * FROM read_files(
                '{volume_path}/{csv_file.name}',
                format => 'csv', header => true, inferSchema => true)""")

        print(f"Created {table_name}")
    cursor.close()

## Data Verification

Quick sanity check to confirm the Databricks table is accessible via the Athena federated connector.

In [ ]:
query = 'SELECT * FROM "lambda:databricks"."default"."paysim_transactions" LIMIT 1'

result = await execute_athena_query(query, s3_location_na_import, database=s3_tables_database)
query_id = result[0].task_id

rows = get_athena_query_results(query_id)
header, row = rows[0], rows[1]
print(f"Federated query returned: type={row[1]}, amount={row[2]}, from={row[3]}");
assert len(rows) == 2, f"Expected 2 rows (1 header + 1 data), got {len(rows)}"


## Data Transformation and Graph Import

In this step, Amazon Athena federates queries against the Databricks Unity Catalog table via 
the Athena-Databricks connector. Two projections are generated:

1. Vertex CSV — Extracts distinct customer IDs (both source and destination) from the 
transaction dataset to create graph nodes.
2. Edge CSV — Maps each transaction as a directed edge between customers, carrying transaction 
attributes (type, amount, balances, fraud flags) as edge properties.

Both projections are written to S3 in Neptune Analytics' CSV import format, cleaned of Athena 
metadata files, and then bulk-imported into the graph.

After completion, the graph contains customer nodes connected by transaction edges — ready for
graph analytics (e.g., fraud ring detection, centrality analysis).

│ **Troubleshooting:** If the Athena federated connector is not configured properly (e.g., the Lambda 
function does not exist or the connector name is incorrect), you will receive a 
`GENERIC_USER_ERROR` with a `ResourceNotFoundException`. Ensure the connector Lambda function is 
deployed and the catalog name in your query (e.g., lambda:databricks) matches the registered 
connector name.


In [ ]:
# Clear import directory
empty_s3_bucket(s3_location_na_import)

# Generate vertex and edge projections from Databricks table
databricks_table_ref=f'"lambda:databricks"."default"."paysim_transactions"'

SOURCE_AND_DESTINATION_CUSTOMERS = f"""
SELECT DISTINCT "~id", 'customer' AS "~label"
FROM (
    SELECT NAMEORIG as "~id" FROM {databricks_table_ref} WHERE NAMEORIG IS NOT NULL
    UNION ALL
    SELECT NAMEDEST as "~id" FROM {databricks_table_ref} WHERE NAMEDEST IS NOT NULL
)
"""

BANK_TRANSACTIONS = f"""
SELECT
    NAMEORIG as "~from",
    NAMEDEST as "~to",
    TYPE AS "~label",
    STEP AS "step:Int",
    AMOUNT AS "amount:Float",
    OLDBALANCEORG AS "oldbalanceOrg:Float",
    NEWBALANCEORIG AS "newbalanceOrig:Float",
    OLDBALANCEDEST AS "oldbalanceDest:Float",
    NEWBALANCEDEST AS "newbalanceDest:Float",
    ISFRAUD AS "isFraud:Int",
    ISFLAGGEDFRAUD AS "isFlaggedFraud:Int"
FROM {databricks_table_ref} WHERE NAMEORIG IS NOT NULL AND NAMEDEST IS NOT NULL
"""

await execute_athena_query(SOURCE_AND_DESTINATION_CUSTOMERS, s3_location_na_import, database=s3_tables_database, polling_interval=15)
await execute_athena_query(BANK_TRANSACTIONS, s3_location_na_import, database=s3_tables_database, polling_interval=15)

# # Remove unnecessary .csv.metadata file generated by Athena. 
empty_s3_bucket(s3_location_na_import, file_extension=".csv.metadata")

task_id = await instance_management.import_csv_from_s3(
        NeptuneGraph.from_config(set_config_graph_id(graph_id)),
        s3_location_na_import)


## Graph Analytics

With the transaction graph loaded, we run community detection to identify clusters of 
customers with dense transaction patterns — a common technique for fraud ring detection.

1. Verify Import — Confirm nodes (customers) and edges (transactions) were loaded correctly.
2. Community Detection — Run the Louvain algorithm on Neptune Analytics to partition the graph 
into communities, writing the result as a community property on each node.
3. Analyze Communities — Query the top 10 communities by size to understand the graph's 
structure and identify unusually large or tightly connected groups for further investigation.



In [ ]:
config = set_config_graph_id(graph_id)
na_graph = NeptuneGraph.from_config(config)

# Verify nodes
all_nodes = na_graph.execute_call("MATCH (n) RETURN n LIMIT 3")
print("Sample Nodes:")
for n in all_nodes:
    print(f"  {n['n']['~id']} ({n['n']['~labels'][0]})")

# Verify edges
all_edges = na_graph.execute_call("MATCH ()-[r]-() RETURN r LIMIT 5")
print("\nSample Edges:")
for e in all_edges:
    r = e["r"]
    print(f"  {r['~start']} --[{r['~type']}, amount: {r['~properties']['amount']}]--> {r['~end']}")


In [ ]:
# Run Louvain algorithm and mutate graph with community property
louvain_result = na_graph.execute_call(
    'CALL neptune.algo.louvain.mutate({iterationTolerance:1e-07, writeProperty:"community"}) '
    'YIELD success AS success RETURN success'
)
print(f"Louvain result: {louvain_result}")

In [ ]:
# Find the top 10 communities by size
top_communities = na_graph.execute_call("""
MATCH (n)
WHERE n.community IS NOT NULL
RETURN n.community AS community, count(*) AS community_size
ORDER BY community_size DESC
LIMIT 10
""")

print("Top 10 Communities:")
print(f"  {'Community ID':>14}  {'Size':>6}")
print(f"  {'─' * 14}  {'─' * 6}")
for c in top_communities:
    print(f"  {c['community']:>14}  {c['community_size']:>6}")


## Conclusion

This notebook demonstrated an end-to-end workflow for federated graph analytics: 
Sourcing transaction data from Databricks via the Athena-Databricks connector, transforming it into a graph-compatible format with Athena, importing it into Neptune Analytics, and running community detection to surface transaction clusters.

This pattern enables teams to leverage existing data in Databricks without data duplication — Athena federates the query at runtime, and Neptune Analytics provides the graph compute layer for analytics that relational engines aren't designed for.